# Complex-to-real channel representations

This notebook confirms that the `Representation` enum converts a complex SAR pass into the
expected stack of real-valued channels. For each representation we plot the original complex
field (magnitude and phase) alongside every channel produced by `convert_into`, so the
identity of each channel can be checked by eye against its definition.

Modules exercised:

- `tools.data.representation.Representation`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Synthetic complex pass

We build a small complex image whose magnitude is a smooth radial bump and whose phase is a
linear ramp. This makes magnitude, phase, real and imaginary parts visually distinct.


In [ ]:
from tools.data.representation import Representation

H, W   = 48, 48
yy, xx = np.mgrid[0:H, 0:W]

cy, cx    = H / 2.0, W / 2.0
radius    = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
magnitude = np.exp(-(radius ** 2) / (2.0 * (H / 4.0) ** 2)).astype(np.float32)

phase     = (2.0 * np.pi * (xx / W) - np.pi).astype(np.float32)
complex_field = (magnitude * np.exp(1j * phase)).astype(np.complex64)

complex_field = complex_field[np.newaxis]
print("complex pass shape:", complex_field.shape, complex_field.dtype)



## Reference views

Magnitude and wrapped phase of the synthetic field define the ground truth that the channel
decompositions must reproduce.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
im0 = axes[0].imshow(np.abs(complex_field[0]), cmap="viridis")
axes[0].set_title("magnitude")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(np.angle(complex_field[0]), cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[1].set_title("phase [rad]")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()



## Channels produced by every representation

For each representation we allocate the output buffer with the documented
`channels_per_pass` and call `convert_into`. The `slot_kinds` list labels each channel.


In [ ]:
representations = [
    Representation.REAL_IMAG,
    Representation.MAG_REAL_IMAG,
    Representation.MAG_ANGLE,
    Representation.MAG_RI_ANGLE,
    Representation.ANGLE_ONLY,
    Representation.MAG_ONLY,
]

for rep in representations:
    cpp = rep.channels_per_pass
    out = np.empty((cpp, H, W), dtype=np.float32)
    rep.convert_into(out, complex_field)

    fig, axes = plt.subplots(1, cpp, figsize=(2.6 * cpp, 2.8), squeeze=False)
    fig.suptitle(f"{rep.value}  (channels_per_pass = {cpp})", y=1.04)

    for c in range(cpp):
        ax = axes[0, c]
        im = ax.imshow(out[c], cmap="magma")
        ax.set_title(rep.slot_kinds[c] + f" [ch {c}]")
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046)

    plt.tight_layout()
    plt.show()



## Numerical cross-check

We verify that the `MAG_RI_ANGLE` channels match direct numpy computations of magnitude,
normalised real, normalised imaginary and angle.


In [ ]:
rep = Representation.MAG_RI_ANGLE
out = np.empty((rep.channels_per_pass, H, W), dtype=np.float32)
rep.convert_into(out, complex_field)

mag       = np.abs(complex_field[0])
safe_mag  = np.where(mag > 0, mag, 1.0)
expected  = [mag, complex_field[0].real / safe_mag, complex_field[0].imag / safe_mag, np.angle(complex_field[0])]

for c, (label, exp) in enumerate(zip(rep.slot_kinds, expected)):
    err = float(np.abs(out[c] - exp).max())
    print(f"ch {c} ({label}): max abs error = {err:.3e}")


## Expected visual outcome

Each representation should display exactly `channels_per_pass` panels whose appearance matches
its `slot_kinds` labels: magnitude panels show the radial bump, phase panels show the wrapped
ramp, and real/imaginary panels show signed lobes. The numerical cross-check should report
errors near machine precision (about 1e-7), confirming the conversion is correct.